# 03 — Feature Engineering

> **Insiders Loyalty Program** · Customer Segmentation Project

---

Géron (Ch. 2): *"Prepare the data to better expose the underlying data patterns to Machine Learning algorithms."*

This notebook covers:
1. RFM feature derivation from raw transactions
2. Outlier inspection (IQR method)
3. Log1p transformation for skewed features
4. StandardScaler preprocessing
5. Sklearn Pipeline preview
6. Before vs after comparison

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from insiders_loyalty_program.config import load_config
from insiders_loyalty_program.data import load_training_frame
from insiders_loyalty_program.features import build_rfm_features

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

config = load_config(PROJECT_ROOT / 'configs' / 'project.toml')
print('Ready.')

## 1. Build RFM Features

In [ ]:
df_raw = load_training_frame(config)
rfm = build_rfm_features(df_raw, config)

rfm_features = [c for c in ['recency_days', 'frequency', 'monetary', 'avg_ticket', 'total_items']
                if c in rfm.columns]

print(f'RFM table shape: {rfm.shape}')
print(f'Features: {rfm_features}')
rfm[rfm_features].describe().round(2)

## 2. Outlier Inspection (IQR method)

Extreme outliers in RFM data are often **real high-value customers**, not errors. We inspect them but do not remove them — instead, the log transform will compress their scale.

In [ ]:
print('Outlier analysis (IQR × 3 threshold):')
print(f'{'Feature':<20} {'Q1':>8} {'Q3':>8} {'IQR':>8} {'Upper':>10} {'Outliers':>10} {'%':>6}')
print('-' * 72)
for feat in rfm_features:
    q1, q3 = rfm[feat].quantile(0.25), rfm[feat].quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 3 * iqr
    n_out = (rfm[feat] > upper).sum()
    pct   = n_out / len(rfm) * 100
    print(f'{feat:<20} {q1:>8.1f} {q3:>8.1f} {iqr:>8.1f} {upper:>10.1f} {n_out:>10} {pct:>5.1f}%')

In [ ]:
# Box plots to visualise outliers
fig, axes = plt.subplots(1, len(rfm_features), figsize=(4 * len(rfm_features), 4))
if len(rfm_features) == 1:
    axes = [axes]

for ax, feat in zip(axes, rfm_features):
    ax.boxplot(rfm[feat].dropna(), notch=False, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(feat.replace('_', ' ').title(), fontsize=9)
    ax.set_xticklabels([])

plt.suptitle('Box Plots — Raw RFM Features (extreme outliers visible)', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / '03_boxplots_raw.png', dpi=120)
plt.show()

## 3. Log1p Transformation

The **log1p (log(1 + x))** transform is the standard approach for compressing right-skewed distributions in retail/financial data:
- Works with zero values (unlike `log`)
- Brings extreme outliers closer to the bulk of data
- Makes K-Means more effective because Euclidean distance is less dominated by large values

This follows the book's recommendation in Ch. 2: *"Feature scaling: many ML algorithms don't perform well when numerical attributes have very different scales."*

In [ ]:
rfm_log = rfm[rfm_features].copy()
rfm_log = rfm_log.apply(np.log1p)

fig, axes = plt.subplots(2, len(rfm_features), figsize=(4 * len(rfm_features), 7))

for j, feat in enumerate(rfm_features):
    # Raw
    axes[0, j].hist(rfm[feat].clip(upper=rfm[feat].quantile(0.99)),
                    bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[0, j].set_title(f'{feat} (raw)', fontsize=9)
    axes[0, j].set_ylabel('Frequency' if j == 0 else '')

    # Log-transformed
    axes[1, j].hist(rfm_log[feat], bins=40, color='darkorange', edgecolor='white', linewidth=0.3)
    axes[1, j].set_title(f'{feat} (log1p)', fontsize=9)
    axes[1, j].set_ylabel('Frequency' if j == 0 else '')

plt.suptitle('Before (blue) vs After (orange) log1p Transform', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / '03_log_transform_comparison.png', dpi=120)
plt.show()
print('Distributions are much more symmetric after log1p — good for K-Means.')

## 4. Skewness Before and After

In [ ]:
skew_raw = rfm[rfm_features].skew().rename('skew_raw')
skew_log = rfm_log.skew().rename('skew_log1p')
skew_df = pd.concat([skew_raw, skew_log], axis=1)
skew_df['reduction_%'] = ((skew_raw - skew_log.abs()) / skew_raw.abs() * 100).clip(lower=0)
print('Skewness (|value| closer to 0 = more symmetric):')
print(skew_df.round(2).to_string())

## 5. Full Preprocessing Pipeline (Scikit-Learn)

Following Géron's Pipeline pattern (Ch. 2): *"Build a Pipeline to make data transformations reproducible and serialisable."*

The pipeline chains:
1. `SimpleImputer` — fill any remaining NaN with the column median
2. `FunctionTransformer(np.log1p)` — compress skewed distributions
3. `StandardScaler` — normalise to zero mean and unit variance

In [ ]:
preprocessing_pipeline = Pipeline(steps=[
    ('impute',    SimpleImputer(strategy='median')),
    ('log',       FunctionTransformer(np.log1p, validate=True)),
    ('scale',     StandardScaler()),
])

X_raw = rfm[rfm_features].values
X_processed = preprocessing_pipeline.fit_transform(X_raw)

X_df = pd.DataFrame(X_processed, columns=rfm_features)
print('Processed feature matrix (first 5 rows):')
print(X_df.head().round(3).to_string())
print(f'\nShape: {X_df.shape}')
print(f'Mean (should be ~0):\n{X_df.mean().round(3).to_string()}')
print(f'Std  (should be ~1):\n{X_df.std().round(3).to_string()}')

## 6. Feature Engineering Summary

| Step | Input | Output | Why |
|---|---|---|---|
| Filter raw transactions | 541,909 rows | ~370,000 rows | Remove cancellations, missing IDs, bad values |
| RFM aggregation | 370,000 rows | ~4,300 rows (one per customer) | Group by CustomerID |
| Log1p transform | Right-skewed features | More symmetric features | K-Means sensitive to scale |
| StandardScaler | Arbitrary ranges | Mean=0, Std=1 | Equal contribution of each feature to distance |

---
**Next notebook:** `04_modeling_and_business_results.ipynb` — train K-Means and profile the clusters.